# 06 — Hybrid Recommendation Engine

In this notebook, we unify the two intelligence layers of PodcastMind into a single, high-performance **Hybrid Recommender System**.

## The Hybrid Advantage
- **Content-Based (Semantic):** Great for precision and new items (Cold Start), but can suffer from "overspecialization" (only suggesting things that look like what you already like).
- **Collaborative Filtering (Behavioral):** Excellent for serendipity and "discovery" (finding non-obvious links), but requires user behavior and can miss obscure items.

By blending these two signals, PodcastMind delivers recommendations that are both **topically relevant** and **behaviorally validated**.

## Objectives
- Load all AI artifacts (FAISS index, ALS model, metadata).
- Implement a **Score Normalization** layer to align semantic and collaborative signals.
- Build the `get_hybrid_recommendations` orchestration engine.
- Implement a **Recommendation Explanation Layer** for improved UX transparency.
- Validate the system across diverse topical domains.

## SECTION 1 — Imports + Artifact Loading

In [ ]:
import pandas as pd
import numpy as np
import faiss
import pickle
import implicit
from pathlib import Path
from sentence_transformers import SentenceTransformer
import warnings

warnings.filterwarnings('ignore')

# Paths
ARTIFACTS_DIR = Path("../artifacts")
PROCESSED_DATA_DIR = Path("../data/processed")
PODCASTS_FILE = PROCESSED_DATA_DIR / "podcasts_subset_20k.csv"

print("Loading artifacts...")

# 1. Load Metadata
with open(ARTIFACTS_DIR / "podcast_metadata.pkl", "rb") as f:
    metadata_mapping = pickle.load(f)

# 2. Load FAISS Index
index = faiss.read_index(str(ARTIFACTS_DIR / "faiss_index.index"))

# 3. Load ALS Model & Mappings
with open(ARTIFACTS_DIR / "als_model.pkl", "rb") as f:
    als_model = pickle.load(f)
with open(ARTIFACTS_DIR / "collaborative_mappings.pkl", "rb") as f:
    collab_mappings = pickle.load(f)

# 4. Load Embedding Model
embed_model = SentenceTransformer("all-MiniLM-L6-v2")

print("All artifacts loaded successfully.")

ModuleNotFoundError: No module named 'implicit'

## SECTION 2 — Base Retrieval Functions

We wrap our existing logic into clean, reusable functions.

In [2]:
def get_semantic_candidates(query, top_k=20):
    """Retrieves top-N semantically similar podcasts."""
    query_vector = embed_model.encode([query], normalize_embeddings=True)
    distances, indices = index.search(query_vector, top_k)
    
    candidates = {}
    for i in range(top_k):
        idx = indices[0][i]
        score = distances[0][i]
        pod_id = metadata_mapping[idx]['podcast_id']
        candidates[pod_id] = {
            'semantic_score': float(score),
            'metadata': metadata_mapping[idx]
        }
    return candidates

def get_collaborative_candidates(podcast_id, top_k=20):
    """Retrieves top-N behaviorally similar podcasts based on co-interests."""
    if podcast_id not in collab_mappings['id_to_idx']:
        return {}
    
    p_idx = collab_mappings['id_to_idx'][podcast_id]
    ids, scores = als_model.similar_items(p_idx, N=top_k+1)
    
    candidates = {}
    for i in range(1, len(ids)): # Skip self
        idx = ids[i]
        score = scores[i]
        pod_id = collab_mappings['idx_to_id'][idx]
        
        # Get metadata from mapping
        # (Note: we assume metadata_mapping order matches our index order)
        candidates[pod_id] = {
            'collaborative_score': float(score),
            'metadata': metadata_mapping[idx]
        }
    return candidates

## SECTION 3 — Score Normalization

Semantic scores (cosine similarity) are between -1 and 1 (usually 0.3-0.8 for real matches).
Collaborative scores (ALS dot products) can be on any scale.

We use **Min-Max Normalization** to force both signals into a 0.0 - 1.0 range so they can be weighted fairly.

In [3]:
def normalize_scores(candidates, score_key):
    if not candidates: return candidates
    
    scores = [c[score_key] for c in candidates.values()]
    min_s, max_s = min(scores), max(scores)
    
    # Avoid division by zero
    if max_s == min_s:
        for pid in candidates: candidates[pod_id][f'norm_{score_key}'] = 1.0
        return candidates
        
    for pid in candidates:
        raw = candidates[pid][score_key]
        candidates[pid][f'norm_{score_key}'] = (raw - min_s) / (max_s - min_s)
    
    return candidates

## SECTION 4 — Hybrid Blending Function

This is the core of our system. It implements a **Weighted Average** of the normalized signals.

In [4]:
def get_hybrid_recommendations(query=None, podcast_id=None, top_k=5, s_weight=0.6, c_weight=0.4):
    """
    Orchestrates the hybrid recommendation process.
    1. Fetch semantic candidates
    2. Fetch collaborative candidates (if podcast_id provided)
    3. Merge and Normalize
    4. Rank by Weighted Score
    """
    # 1. Candidate Retrieval
    semantic_pool = get_semantic_candidates(query, top_k=50) if query else {}
    collab_pool = get_collaborative_candidates(podcast_id, top_k=50) if podcast_id else {}
    
    # 2. Normalization
    semantic_pool = normalize_scores(semantic_pool, 'semantic_score')
    collab_pool = normalize_scores(collab_pool, 'collaborative_score')
    
    # 3. Merging (Union of IDs)
    all_ids = set(semantic_pool.keys()) | set(collab_pool.keys())
    
    hybrid_results = []
    for pid in all_ids:
        # Get base metadata
        metadata = semantic_pool.get(pid, collab_pool.get(pid))['metadata']
        
        # Get normalized scores (default to 0 if not present in that pool)
        s_score = semantic_pool.get(pid, {}).get('norm_semantic_score', 0.0)
        c_score = collab_pool.get(pid, {}).get('norm_collaborative_score', 0.0)
        
        # Compute weighted blend
        blended_score = (s_score * s_weight) + (c_score * c_weight)
        
        hybrid_results.append({
            'podcast_id': pid,
            'title': metadata['title'],
            'categories': metadata['categories'],
            'semantic_score': s_score,
            'collaborative_score': c_score,
            'blended_score': blended_score
        })
    
    # 4. Reranking
    final_df = pd.DataFrame(hybrid_results).sort_values('blended_score', ascending=False)
    
    # Remove the target podcast if it's in the results (for podcast-to-podcast recs)
    if podcast_id:
        final_df = final_df[final_df['podcast_id'] != podcast_id]
        
    return final_df.head(top_k)

print("Hybrid engine ready.")

Hybrid engine ready.


## SECTION 5 — Recommendation Explanation Layer

Explainability increases user trust. We generate a simple textual reason for each recommendation based on which engine contributed the most.

In [5]:
def generate_explanation(row):
    if row['semantic_score'] > 0.8 and row['collaborative_score'] > 0.8:
        return "A perfect match! It is semantically relevant and popular among similar listeners."
    elif row['semantic_score'] > row['collaborative_score']:
        return f"Topically similar to your interests in {row['categories'].split(',')[0]}."
    else:
        return "Listeners who enjoy similar shows also explored this podcast."

print("Explanation layer ready.")

Explanation layer ready.


## SECTION 6 — Hybrid Retrieval Validation

Let's test the system with a real scenario: A user who likes **Business and Startups**.

In [6]:
query = "Startups, entrepreneurship and venture capital"
results = get_hybrid_recommendations(query=query, top_k=5)

results['explanation'] = results.apply(generate_explanation, axis=1)

print(f"QUERY: {query}")
display(results[['title', 'categories', 'blended_score', 'explanation']])

QUERY: Startups, entrepreneurship and venture capital


,title,categories,blended_score,explanation
9,Metro Startup Launcher,"business-investing, business",0.600000,Topically similar to your interests in busines...
28,Austinpreneur,"business-investing, business-entrepreneurship,...",0.518574,Topically similar to your interests in busines...
17,Fast Growth Business,"business-careers, business, business-investing",0.512301,Topically similar to your interests in busines...
0,Entrepreneurship,"science, education-courses, education, science...",0.421175,Topically similar to your interests in science.
41,Project A Podcast,"business-investing, business, business-management",0.406032,Topically similar to your interests in busines...


## SECTION 7 — Cold Start Handling

If we don't have a specific `podcast_id` (e.g., a brand new user), the system dynamically shifts to 100% semantic weighting.

In [7]:
print("--- Cold Start Scenario (Query Only) ---")
cold_start = get_hybrid_recommendations(query="Space exploration and Mars", s_weight=1.0, c_weight=0.0)
display(cold_start[['title', 'categories', 'blended_score']].head(3))

--- Cold Start Scenario (Query Only) ---


,title,categories,blended_score
24,Mars Ascend-Humans to Mars and Beyond,"science-social-sciences, society-culture, scie...",1.000000
15,Lessons from Mars,"business-management, business-careers, business",0.869233
46,Are We There Yet?,"science-natural-sciences, science-astronomy, s...",0.828917


## SECTION 8 — Save Hybrid Configuration

In [8]:
hybrid_config = {
    'default_semantic_weight': 0.7,
    'default_collaborative_weight': 0.3,
    'min_semantic_threshold': 0.4,
    'embedding_model': 'all-MiniLM-L6-v2'
}

with open(ARTIFACTS_DIR / "hybrid_config.pkl", "wb") as f:
    pickle.dump(hybrid_config, f)

print(f"Configuration saved to {ARTIFACTS_DIR}")

Configuration saved to ..\artifacts


## SECTION 9 — Final Architecture Summary

The PodcastMind recommendation pipeline is now complete:

1. **Input:** User Query or Interaction ID.
2. **Candidate Retrieval:** 
   - FAISS searches the 384D embedding space.
   - ALS retrieves behavioral neighbors.
3. **Scoring:** Signals are normalized and weighted (Hybrid Blend).
4. **Ranking:** Final list is sorted and deduplicated.
5. **Explanation:** Contextual reasons are attached to each result.
6. **Output:** A high-quality list of podcasts ready for the backend API.